# 03 · LLM-assisted meta descriptions with Claude, plus guardrails

Companion notebook to the MLforSEO tutorial *Automatically Generate Meta Descriptions Using Python: from BERT to LLMs*.

A hosted LLM writes the most natural-sounding descriptions of all the options, and it can follow instructions about tone, the target query and length.
It also costs money per page, sends your content to a third party, and can **hallucinate**: a number, a feature or a promise that isn't on the page.
So this notebook is built around three habits:

1. **Estimate cost before you run anything.** `DRY_RUN = True` by default: it builds the prompts and prices them without calling the model.
2. **Ground the prompt** in the page text, and tell the model what to do when the page doesn't support a claim.
3. **Check every output** (length, numbers and names not on the page, missing query, duplicates, drift from the page), then have a human approve it.

You need an Anthropic API key from [console.anthropic.com](https://console.anthropic.com/). In Colab, store it under 🔑 *Secrets* as `ANTHROPIC_API_KEY`.
Never paste keys into a notebook you share.

> Prefer OpenAI? The pattern is the same. Our [Fuzzy Booster tutorial](https://www.mlforseo.com/machine-learning-implementation-guides/onpage-seo/how-to-automatically-optimize-your-seo-metadata-with-fuzzywuzzy-and-openai-in-google-colab/) shows the OpenAI client in Colab.

In [1]:
# Install anything that's missing (works in Google Colab and locally)
import importlib.util, subprocess, sys

PACKAGES = {'anthropic': 'anthropic', 'sentence_transformers': 'sentence-transformers', 'trafilatura': 'trafilatura', 'pandas': 'pandas'}  # import name -> pip name
missing = [pip for mod, pip in PACKAGES.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
print("Ready. Installed now:", missing or "nothing, all present")

Ready. Installed now: nothing, all present


## 1. Load your pages

In [2]:
import pandas as pd

# OPTION A (the 2021 way): a CSV you exported from Screaming Frog, Sitebulb, etc.
#   Column A = URL, Column B = the page's main content. Title and Top query are optional extras.
INPUT_FILE = "inputdata.csv"

# OPTION B: no crawler export? Put one URL per line in urls.txt and set this to True.
#   trafilatura downloads each page and pulls out the main text and the title for you.
SCRAPE_FROM_URL_LIST = False
URL_LIST_FILE = "urls.txt"

# In Colab, upload your file first (folder icon on the left), or uncomment:
# from google.colab import files; files.upload()

if SCRAPE_FROM_URL_LIST:
    import trafilatura
    rows = []
    for url in [u.strip() for u in open(URL_LIST_FILE) if u.strip()]:
        html = trafilatura.fetch_url(url)
        text = trafilatura.extract(html) if html else None
        meta = trafilatura.extract_metadata(html) if html else None
        rows.append({"URL": url, "Content": text or "", "Title": getattr(meta, "title", "") or ""})
    df = pd.DataFrame(rows)
else:
    # error_bad_lines=False (2021) was removed in pandas 2.0; on_bad_lines="skip" replaces it
    df = pd.read_csv(INPUT_FILE, encoding="utf-8-sig", on_bad_lines="skip")
    df = df.rename(columns={df.columns[0]: "URL", df.columns[1]: "Content"})

for col in ["Title", "Top query"]:
    if col not in df.columns:
        df[col] = ""
df = df.fillna("")
df["Words"] = df["Content"].str.split().str.len()
df[["URL", "Title", "Top query", "Words"]]

,URL,Title,Top query,Words
0,https://www.mlforseo.com/machine-learning-tuto...,"What Fuzzy Matching Is, and Where It Fits in SEO",what is fuzzy matching,2349
1,https://www.mlforseo.com/machine-learning-tuto...,Topic Modelling for SEO with BERTopic,bertopic seo,1571
2,https://www.mlforseo.com/machine-learning-impl...,How to do keyword clustering with KeyBERT,keybert keyword clustering,2212
3,https://www.mlforseo.com/machine-learning-impl...,How to Automatically Optimize your SEO Metadat...,automate meta descriptions,1938
4,https://www.mlforseo.com/glossary-term/bert-bi...,BERT (Bidirectional Encoder Representations fr...,what is bert,233


## 2. Settings

- `MODEL`: `claude-opus-5-5` is Anthropic's current default model. For thousands of short descriptions you may want to test a cheaper model such as
  `claude-haiku-4-5` on a sample and compare quality. Prices below are per million tokens and were correct in October 2026; check
  [the pricing page](https://www.anthropic.com/pricing) before a big run.
- `EFFORT`: writing a 150-character summary is an easy task, so `"low"` keeps thinking (and cost) down.
- `MAX_WORDS`: how much of each page to send. More words cost more and rarely improve a two-sentence summary.

In [3]:
DRY_RUN   = True                 # True = build prompts and estimate cost only, no API calls
MODEL     = "claude-opus-5-5"
EFFORT    = "low"
MAX_WORDS = 800

PRICES = {  # USD per 1M tokens (input, output), October 2026
    "claude-opus-5-5": (4.00, 20.00),
    "claude-sonnet-5-5": (2.00, 10.00),
    "claude-haiku-4-5": (1.00, 5.00),
}

BRAND_VOICE = "Friendly, clear and practical. UK English. No hype words like 'ultimate', 'revolutionary' or 'unlock'."

SYSTEM_PROMPT = f"""You write meta descriptions for web pages.

Rules:
- Use only facts that are stated in the page text. Do not add numbers, prices, features, names or claims that are not in it.
- One or two sentences, 120-155 characters in total.
- If a target query is given and the page is genuinely about it, use the query (or a natural variant) once.
- Summarise what the reader will get from the page. Inspire curiosity for informational pages; do not give the whole answer away.
- Voice: {BRAND_VOICE}
- Reply with the meta description only: plain text, no quotes, no labels, no emojis.
- If the page text is too thin to describe, reply with exactly: NEEDS HUMAN"""

def build_user_message(row):
    page = " ".join(str(row["Content"]).split()[:MAX_WORDS])
    return (f"URL: {row['URL']}\nPage title: {row['Title']}\nTarget query: {row['Top query'] or 'none'}\n\n"
            f"<page_text>\n{page}\n</page_text>\n\nWrite the meta description.")

df["prompt"] = df.apply(build_user_message, axis=1)

if not DRY_RUN:
    import os, anthropic
    if "ANTHROPIC_API_KEY" not in os.environ:
        try:  # Google Colab: read the key from Secrets
            from google.colab import userdata
            os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
        except ImportError:
            pass
    client = anthropic.Anthropic()
print(SYSTEM_PROMPT, "\n\n--- first user message ---\n", df["prompt"].iloc[0][:800], "...")

You write meta descriptions for web pages.

Rules:
- Use only facts that are stated in the page text. Do not add numbers, prices, features, names or claims that are not in it.
- One or two sentences, 120-155 characters in total.
- If a target query is given and the page is genuinely about it, use the query (or a natural variant) once.
- Summarise what the reader will get from the page. Inspire curiosity for informational pages; do not give the whole answer away.
- Voice: Friendly, clear and practical. UK English. No hype words like 'ultimate', 'revolutionary' or 'unlock'.
- Reply with the meta description only: plain text, no quotes, no labels, no emojis.
- If the page text is too thin to describe, reply with exactly: NEEDS HUMAN 

--- first user message ---
 URL: https://www.mlforseo.com/machine-learning-tutorials/fuzzy-matching/what-fuzzy-matching-is-and-where-it-fits-in-seo/
Page title: What Fuzzy Matching Is, and Where It Fits in SEO
Target query: what is fuzzy matching

<page_text

## 3. Estimate the cost first

Offline, we approximate tokens as characters ÷ 4 (rough, but fine for a budget). With a key set, `messages.count_tokens` gives the exact input count.
The output side is small: about 40 to 60 tokens per description, plus any thinking the model does.

In [4]:
in_price, out_price = PRICES[MODEL]
approx_in = sum(len(SYSTEM_PROMPT) + len(p) for p in df["prompt"]) / 4
approx_out = 150 * len(df)          # generous allowance per page for the answer plus a little thinking
est = approx_in / 1e6 * in_price + approx_out / 1e6 * out_price
print(f"{len(df)} pages · ~{approx_in:,.0f} input tokens · ~{approx_out:,.0f} output tokens")
print(f"Estimated cost on {MODEL}: ${est:.4f}  (≈ ${est / len(df) * 1000:.2f} per 1,000 pages)")

if not DRY_RUN:
    exact = sum(client.messages.count_tokens(model=MODEL, system=SYSTEM_PROMPT,
                                             messages=[{"role": "user", "content": p}]).input_tokens
                for p in df["prompt"])
    print(f"Exact input tokens: {exact:,}  → input cost ${exact / 1e6 * in_price:.4f}")

5 pages · ~6,767 input tokens · ~750 output tokens
Estimated cost on claude-opus-5-5: $0.0421  (≈ $8.41 per 1,000 pages)


## 4. Generate

Each page is one request. A few details that matter:
- `fallbacks="default"` (beta) means that if the model's safety classifier declines a request, which is rare for meta descriptions but possible, the API retries on a recommended fallback model instead of returning nothing.
- We check `stop_reason` before reading the text, and we keep `max_tokens` generous so thinking never cuts the answer off.
- For thousands of pages, the [Message Batches API](https://docs.anthropic.com/en/docs/build-with-claude/batch-processing) runs the same requests asynchronously at half the price.

In [5]:
def claude_meta(user_message):
    response = client.beta.messages.create(
        model=MODEL,
        max_tokens=2000,
        system=SYSTEM_PROMPT,
        output_config={"effort": EFFORT},
        betas=["server-side-fallback-2026-07-01"],
        fallbacks="default",
        messages=[{"role": "user", "content": user_message}],
    )
    if response.stop_reason == "refusal":
        return "NEEDS HUMAN", response.usage
    text = "".join(b.text for b in response.content if b.type == "text").strip().strip('"')
    return text, response.usage

if DRY_RUN:
    df["Claude meta"] = ""
    print("DRY_RUN is on: no API calls made. Set DRY_RUN = False in the settings cell to generate.")
else:
    texts, in_tok, out_tok = [], 0, 0
    for p in df["prompt"]:
        t, usage = claude_meta(p)
        texts.append(t); in_tok += usage.input_tokens; out_tok += usage.output_tokens
    df["Claude meta"] = texts
    print(f"Actual usage: {in_tok:,} in / {out_tok:,} out → ${in_tok / 1e6 * in_price + out_tok / 1e6 * out_price:.4f}")

df[["URL", "Claude meta"]]

DRY_RUN is on: no API calls made. Set DRY_RUN = False in the settings cell to generate.


,URL,Claude meta
0,https://www.mlforseo.com/machine-learning-tuto...,
1,https://www.mlforseo.com/machine-learning-tuto...,
2,https://www.mlforseo.com/machine-learning-impl...,
3,https://www.mlforseo.com/machine-learning-impl...,
4,https://www.mlforseo.com/glossary-term/bert-bi...,


## 5. Guardrails: check every description

In [6]:
import re

MIN_CHARS, MAX_CHARS = 70, 155   # Google truncates by pixel width (~920px desktop); ~155 characters is a safe proxy

def check_description(desc, source_text, top_query=""):
    """Flags problems a human should look at. It does not prove a description is good."""
    flags = []
    n = len(desc)
    if n == 0:
        flags.append("EMPTY")
    elif n < MIN_CHARS:
        flags.append(f"SHORT ({n})")
    elif n > MAX_CHARS:
        flags.append(f"LONG ({n})")
    # numbers that appear in the description but nowhere in the page = classic hallucination
    src_numbers = set(re.findall(r"\d+(?:[.,]\d+)?", source_text))
    new_numbers = [x for x in re.findall(r"\d+(?:[.,]\d+)?", desc) if x not in src_numbers]
    if new_numbers:
        flags.append("NUMBER NOT IN PAGE: " + ", ".join(new_numbers))
    # capitalised words (names, brands, tools) that never appear in the page
    src_lower = source_text.lower()
    new_names = [w for w in re.findall(r"\b[A-Z][A-Za-z0-9]{2,}\b", desc)[1:] if w.lower() not in src_lower]
    if new_names:
        flags.append("NAME NOT IN PAGE: " + ", ".join(sorted(set(new_names))))
    if top_query:
        q_words = [w for w in re.findall(r"\w+", top_query.lower()) if len(w) > 3]
        if q_words and not any(w in desc.lower() for w in q_words):
            flags.append("TOP QUERY MISSING")
    return "; ".join(flags) or "OK"

In [7]:
# Duplicate descriptions are what we're trying to get rid of in the first place, so check for them
def flag_duplicates(frame, col):
    dupes = frame[col].str.strip().str.lower().duplicated(keep=False) & frame[col].str.strip().ne("")
    frame.loc[dupes, col + " check"] = frame.loc[dupes, col + " check"].replace("OK", "") + "; DUPLICATE"
    frame[col + " check"] = frame[col + " check"].str.strip("; ")
    return frame

In [8]:
from sentence_transformers import SentenceTransformer, util

df["Approved"] = ""   # a human fills this in. Nothing goes live without it.
if DRY_RUN:
    df["Claude meta check"], df["Claude meta ~page"] = "not generated (DRY_RUN)", None
    print("DRY_RUN is on, so there is nothing to check yet.")
else:
    df["Claude meta check"] = [check_description(d, s, q) if d != "NEEDS HUMAN" else "NEEDS HUMAN"
                               for d, s, q in zip(df["Claude meta"], df["Content"], df["Top query"])]
    df = flag_duplicates(df, "Claude meta")

    embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
    page_vecs = embedder.encode(df["Content"].str[:5000].tolist(), convert_to_tensor=True, normalize_embeddings=True)
    desc_vecs = embedder.encode(df["Claude meta"].tolist(), convert_to_tensor=True, normalize_embeddings=True)
    df["Claude meta ~page"] = util.cos_sim(desc_vecs, page_vecs).diagonal().cpu().numpy().round(2)

df[["URL", "Claude meta", "Claude meta check", "Claude meta ~page"]]

DRY_RUN is on, so there is nothing to check yet.


,URL,Claude meta,Claude meta check,Claude meta ~page
0,https://www.mlforseo.com/machine-learning-tuto...,,not generated (DRY_RUN),None
1,https://www.mlforseo.com/machine-learning-tuto...,,not generated (DRY_RUN),None
2,https://www.mlforseo.com/machine-learning-impl...,,not generated (DRY_RUN),None
3,https://www.mlforseo.com/machine-learning-impl...,,not generated (DRY_RUN),None
4,https://www.mlforseo.com/glossary-term/bert-bi...,,not generated (DRY_RUN),None


## 6. Save for review

Review the CSV, fix or reject flagged rows, mark `Approved`, and only then import the descriptions into your CMS
(for WordPress, Yoast and Rank Math both support bulk CSV import of SEO meta).

In [9]:
df.drop(columns=["Content", "prompt"]).to_csv("output_claude.csv", index=False)
print("Saved output_claude.csv with", len(df), "rows")

Saved output_claude.csv with 5 rows
